# 1. Data Preparation
The goal of this notebook is to extract data from the extracted Excel files based on a ReviewIndex. The ReviewIndex should contain inclusion/exclusion decisions as well as ID columns for us to associate the extracted data with the overall meta-data. This data should then be reformatted for easy input into the Stan Modelling process. Below is a list of values that we need to extract for Stan modelling.

1. n_row (int): Number of studies included in the analysis, each will be represented as a row in the total/severe matrix.
2. n_col (int): Number of serotype-prior exposure status combinations, which will be the number of columns in the total/severe matrix.
3. n_reg (int): Number of regions in the analysis
4. n_outcomes (int): Number of outcomes in the analysis i.e. number of non-zero cells in the total matrix
5. total (matrix; n_row x n_col): Number of dengue cases for study in row $i$ with serotype-prior exposure in col $j$
6. severe (matrix; n_row x n_col): Number of severe cases for study in row $i$ with serotype-prior exposure in col $j$
7. scenario_indices (array int; n_reg x n_col): Lookup table to check the index of each scenario (used in fitting random effect SD sigma)
8. row_indices (array int; n_outcomes): Row index for each outcome included in the analysis
9. col_indices (array int; n_outcomes): Column index for each outcome included in the analysis
10. reg_indices (array int; n_outcomes): Region index for each outcome included in the analysis
11. region_vals (array int; n_row): Region index of study in index $i$ (also corresponding to row $i$ in the total matrix)
12. num_results (array int; n_row): Number of outcomes for study index $i$ (i.e. non-zero rows in row $i$ of the total matrix)
13. num_scenarios (int): Number of scenarios included in the analysis (combination of serotype-prior exposure-region with at least 2 studies included)
14. n_sero_prior (int): Number of serotype-prior exposure combinations included in the analysis
15. reg_1_scen (int): Number of scenarios in region 1
16. reg_2_scen (int): Number of scenarios in region 2
17. char_matrix (array int; n_outcomes x (n_reg + n_sero_prior - 1)): Characteristic matrix for each outcome included in the analysis. This will be multiplied to beta values to then get effects to add to the intercept.


In [1]:
library(dplyr)
library(tidyverse)
library(ggplot2)
library(readxl)
library(writexl)
options(repr.matrix.max.cols=30, repr.matrix.max.rows=80)

min_cases = 5


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ readr     2.2.0
✔ ggplot2   4.0.2     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.1     ✔ tidyr     1.3.2
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors


In [2]:
base_dir = getwd() #Where the notebook is in
setwd("..")
index_dir = getwd() #Where the review index is in
setwd(base_dir)
data_dir = file.path(index_dir, "Data") #Where the extracted Excel data files are
output_dir = file.path(base_dir, "Processed Data", "Main Results") #Where we will output the processed data to be used as Stan input

save_output = FALSE

In [3]:
#Regions part of the analysis
regions = c("Americas", "Asia")
remove_regions = c("Africa") #Regions to remove - currently only the single study from Africa

#Reference class for the logistic regression
ref_region = "Asia"
ref_serotype_exposure = "Secondary-DENV2"
ref_scenario = paste0(ref_region, "-", ref_serotype_exposure)

#Generate combinations of prior exposure and serotype
serotype_exposure = expand.grid(Serotype = paste0("DENV", 1:4), PriorExp = c("Unknown", "Primary", "Secondary")) %>%
                        mutate(SeroPriorExp = paste0(PriorExp, "-", Serotype)) %>% mutate(ColIndex = 1:nrow(.))

region_df = data.frame(Region = regions) %>% mutate(RegIndex = 1:nrow(.))

scenarios = expand.grid(SeroPriorExp = serotype_exposure$SeroPriorExp, Region = regions) %>% 
                mutate(Scenario = paste0(Region, "-", SeroPriorExp))
scenarios

      SeroPriorExp   Region                 Scenario
1    Unknown-DENV1 Americas   Americas-Unknown-DENV1
2    Unknown-DENV2 Americas   Americas-Unknown-DENV2
3    Unknown-DENV3 Americas   Americas-Unknown-DENV3
4    Unknown-DENV4 Americas   Americas-Unknown-DENV4
5    Primary-DENV1 Americas   Americas-Primary-DENV1
6    Primary-DENV2 Americas   Americas-Primary-DENV2
7    Primary-DENV3 Americas   Americas-Primary-DENV3
8    Primary-DENV4 Americas   Americas-Primary-DENV4
9  Secondary-DENV1 Americas Americas-Secondary-DENV1
10 Secondary-DENV2 Americas Americas-Secondary-DENV2
11 Secondary-DENV3 Americas Americas-Secondary-DENV3
12 Secondary-DENV4 Americas Americas-Secondary-DENV4
13   Unknown-DENV1     Asia       Asia-Unknown-DENV1
14   Unknown-DENV2     Asia       Asia-Unknown-DENV2
15   Unknown-DENV3     Asia       Asia-Unknown-DENV3
16   Unknown-DENV4     Asia       Asia-Unknown-DENV4
17   Primary-DENV1     Asia       Asia-Primary-DENV1
18   Primary-DENV2     Asia       Asia-Primary

## Data Reading
In this section, we read the data in from the Excel files. First, we read in the index, which will then give us the path to the extracted data.

In [4]:
#We read in the review index file 
index_df = read_excel(file.path(index_dir, "ReviewIndex_Final.xlsx"), sheet = "Main") #Read the main sheet containing the index information 
reg_mapper = read_excel(file.path(index_dir, "ReviewIndex_Final.xlsx"), sheet = "CountryRegions") #Read in the sheet to map countries to their geographic region

#Get only the included studies
include_df = index_df %>% filter(FinalDecision %in% c("Include"))

#If the CovidenceID is blank, use the CovidenceID_JanUpdate value (the JanUpdate IDs are new ones from the updated Covidence review we made)
include_df = include_df %>% mutate(CovidenceID = ifelse(is.na(CovidenceID), CovidenceID_JanUpdate, CovidenceID))

#Create a DataFrame with the filename pointing to the Excel file with the data
filename_df = include_df %>% select(ID, CovidenceID, Name, Country, SpecificLocation, WHOClass, InfectionTypes) %>% 
                mutate(File = paste0(ID, "-", Name, ".xlsx")) %>% 
                merge(reg_mapper, by = "Country") #Label the study with the region the study country is in

In [5]:
#Function to read in the data
read_file = function(curr_row){
    #Filename of the Excel file to read
    curr_filename = curr_row["File"]
    
    #Vector of prior exposure statuses given by the study, which correspond with Excel sheets to read
    curr_sheets = curr_row["InfectionTypes"] %>% str_split(pattern = ", ") %>% unlist
    
    #Check for excluded sheets or those potentially mislabelled, and print them out as a warning
    for(temp in curr_sheets){
        if(!temp %in% c("Unspecified", "Primary", "Secondary", "ND")){
            print(temp)
        }
    }
    
    #We exclude sheets outside Unspecified, Primary, Secondary, and ND
    curr_sheets = curr_sheets[curr_sheets %in% c("Unspecified", "Primary", "Secondary", "ND")]
    
    #Start reading sheet
    read_sheet = function(curr_sheet){
        #Read the Excel file and rename the first column to be the serotype
        #curr_data = suppressMessages({read_excel(file.path(data_dir, curr_filename), sheet = curr_sheet) %>% rename(Serotype = `...1`)})
        
        curr_data = read_excel(file.path(data_dir, curr_filename), sheet = curr_sheet) %>% rename(Serotype = `...1`)
        #Check which severity class type the study is in: (1) 1997-type, (2) 2009-type, (3) Hospitalisation
        #This is based on the columns of the data set (DF/DHF/DSS is 1997-type, Dw/woWS/SD is 2009-type, and Hospitalisation is hospitalised or not)
        class_type = ifelse("Hospitalised" %in% colnames(curr_data), "Hospitalisation", ifelse("DF" %in% colnames(curr_data), "1997-type", "2009-type"))
        
        #Check if the study has any asymptomatic cases extracted and print a message if it does
        asymp_checker = curr_data %>% pivot_longer(-Serotype, names_to = "SeverityClass", values_to = "Count") %>% 
                        filter((SeverityClass == "Asymptomatic") & (Count > 0))
        if(nrow(asymp_checker) > 0){print(paste0("Asymptomatic in ", curr_filename))}
        
        #Format the data
        # suppressMessages({curr_data = curr_data %>% pivot_longer(-Serotype, names_to = "SeverityClass", values_to = "Count") %>%
        #             filter(SeverityClass != "Asymptomatic") %>% #Remove asymptomatic cases
        #             mutate(Severity = ifelse(SeverityClass %in% c("DHF", "DSS", "DHF/DSS", "Hospitalised", "SD"), "Severe", "NonSevere")) %>% #Label cases as severe or non-severe
        #             select(-SeverityClass) %>% replace_na(list(Count = 0)) %>% #Change all the NAs as 0s
        #             group_by(Serotype, Severity) %>% summarise(Count = sum(Count)) %>% ungroup %>% #Sum counts of severe and non-severe
        #             pivot_wider(values_from = Count, names_from = Severity) %>% #Pivot to have separate severe, non-severe columns
        #             mutate(N = Severe + NonSevere, ID = curr_row["ID"], CovidenceID = curr_row["CovidenceID"], #Add some columns to label the data 
        #                   PriorExp = curr_sheet, SeveritySystem = curr_row["WHOClass"], #Label with Prior exposure based on sheet name, and severity class system
        #                   SeverityType = class_type, Region = curr_row["Region"], Name = curr_filename)}) #Label the row of data
              #Format the data
        curr_data = curr_data %>% pivot_longer(-Serotype, names_to = "SeverityClass", values_to = "Count") %>%
                    filter(SeverityClass != "Asymptomatic") %>% #Remove asymptomatic cases
                    mutate(Severity = ifelse(SeverityClass %in% c("DHF", "DSS", "DHF/DSS", "Hospitalised", "SD"), "Severe", "NonSevere")) %>% #Label cases as severe or non-severe
                    select(-SeverityClass) %>% replace_na(list(Count = 0)) %>% #Change all the NAs as 0s
                    group_by(Serotype, Severity) %>% summarise(Count = sum(Count)) %>% ungroup %>% #Sum counts of severe and non-severe
                    pivot_wider(values_from = Count, names_from = Severity) %>% #Pivot to have separate severe, non-severe columns
                    mutate(N = Severe + NonSevere, ID = curr_row["ID"], CovidenceID = curr_row["CovidenceID"], #Add some columns to label the data 
                          PriorExp = curr_sheet, SeveritySystem = curr_row["WHOClass"], #Label with Prior exposure based on sheet name, and severity class system
                          SeverityType = class_type, Region = curr_row["Region"], Name = curr_filename) #Label the row of data
        return(curr_data)
    }
    #Read all the sheets of the current file
    to_ret = do.call(rbind, lapply(curr_sheets, read_sheet))
    return(to_ret)
}

In [6]:
#Generate the data set by calling the processing read_file function on each row
data_set = do.call(rbind, apply(filename_df, 1, read_file))

New names:
• `` -> `...1`
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by Serotype and Severity.
ℹ Output is grouped by Serotype.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(Serotype, Severity))` for ]8;;x-r-help:dplyr::dplyr_byper-operation grouping]8;; instead.
New names:
• `` -> `...1`
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by Serotype and Severity.
ℹ Output is grouped by Serotype.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(Serotype, Severity))` for ]8;;x-r-help:dplyr::dplyr_byper-operation grouping]8;; instead.
New names:
• `` -> `...1`
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by Serotype and Severity.
ℹ Output is grouped by Serotype.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(Serotype, Severity))` for ]8;;x-r-help:dplyr::dplyr_byper-op

In [7]:
#Counting the number of included studies overall. Note that we subtract 1 from the total since the separate Asian and American arms of the study by Dussart et al. are counted
#individually here 
all_inc_ids = filename_df %>% pull(CovidenceID) %>% unique
all_inc_ids[!(all_inc_ids %in% (data_set %>% filter(N >= min_cases) %>% pull(CovidenceID) %>% unique))]
length(all_inc_ids)

[1] 74

In [8]:
all_inc_ids

 [1] "#66490 - Fiora 2024"        "#1923 - Hanna 1998"         "#1924 - Hanna 2001"         "#66352 - Rahim 2023"        "#2880 - Ly 2022"           
 [6] "#661 - Burattini 2016"      "#3017 - Martins 2014"       "#1770 - Guerra-Gomes 2017"  "#4072 - Rosa 2000"          "#617 - Braga 2016"         
[11] "#1503 - Feres 2006"         "#395 - Azevedo 2019"        "#66248 - Estofolete 2023"   "#1776 - Guilarde 2008"      "#1379 - Dussart 2012a"     
[16] "#1379 - Dussart 2012b"      "#3677 - Perdomo-Celis 2017" "#3478 - Ocazionez 2006"     "#1469 - Falconar 2012"      "#365 - Aubry 2019"         
[21] "#1399 - Effler 2005"        "#4302 - Sharma 2020"        "#3868 - Racherla 2018"      "#3246 - Mukherjee 2022"     "#66164 - Jadeja 2022"      
[26] "#3936 - Rao 2018"           "#819 - Chakravarti 2010"    "#3076 - Mehta 2018"         "#2299 - Kallol 2016"        "#3739 - Poeranto 2016"     
[31] "#514 - BetiErnawati 2021"   "#66328 - Nainggolan 2023"   "#5125 - Wardhani 2017"      "#1252 -

## Extra Exclusions
In this section, we manually remove some studies after some review. Studies for Africa are removed from the analysis (since we lack studies for the region) and we have to take studies out if all the scenarios they contribute studies to are excluded from the review. In the initial search, there was only one study manually removed, which was "#4786 - Thomas 2008" since it was the only study giving Hospitalisation data for Primary DENV-2 and -4 in the Americas (and thus it cannot be used to generate pooled estimates). Since all of the data it contributes is excluded, the study itself is not part of the analysis.

In [9]:
remove_ids = c("#4786 - Thomas 2008")
data_set = data_set %>% filter(!(Region %in% remove_regions)) #Remove certain regions that we have listed for removal - Africa
data_set = data_set %>% filter(!(CovidenceID %in% remove_ids)) #Remove listed IDs
data_set

# A tibble: 464 × 11
   Serotype NonSevere Severe     N ID    CovidenceID         PriorExp    SeveritySystem  SeverityType    Region   Name                  
 * <chr>        <dbl>  <dbl> <dbl> <chr> <chr>               <chr>       <chr>           <chr>           <chr>    <chr>                 
 1 DENV1           25      0    25 0315  #66490 - Fiora 2024 Unspecified 2009Binary      2009-type       Americas 0315-Fiora_2024a.xlsx 
 2 DENV2            0      0     0 0315  #66490 - Fiora 2024 Unspecified 2009Binary      2009-type       Americas 0315-Fiora_2024a.xlsx 
 3 DENV3            0      0     0 0315  #66490 - Fiora 2024 Unspecified 2009Binary      2009-type       Americas 0315-Fiora_2024a.xlsx 
 4 DENV4            0      0     0 0315  #66490 - Fiora 2024 Unspecified 2009Binary      2009-type       Americas 0315-Fiora_2024a.xlsx 
 5 DENV1           25      0    25 0315  #66490 - Fiora 2024 Unspecified Hospitalisation Hospitalisation Americas 0315-Fiora_2024b.xlsx 
 6 DENV2            

In [10]:
data_set %>% filter(N >= min_cases) %>% arrange(CovidenceID) %>% pull(CovidenceID) %>% unique

 [1] "#1252 - Dewi 2014"          "#1347 - Dumre 2017"         "#1379 - Dussart 2012a"      "#1379 - Dussart 2012b"      "#1385 - Duyen 2011"        
 [6] "#1399 - Effler 2005"        "#1461 - Fakeeh 2001"        "#1469 - Falconar 2012"      "#1503 - Feres 2006"         "#1583 - Fried 2010"        
[11] "#1695 - GomezDantes 1988"   "#1770 - Guerra-Gomes 2017"  "#1776 - Guilarde 2008"      "#1801 - Gupta 2015"         "#1923 - Hanna 1998"        
[16] "#1924 - Hanna 2001"         "#1944 - Harris 2000"        "#2084 - Huang 2020"         "#2299 - Kallol 2016"        "#2326 - Kao 2016"          
[21] "#2379 - Kerdpanich 2021"    "#2498 - Kosasih 2016"       "#2880 - Ly 2022"            "#3017 - Martins 2014"       "#3076 - Mehta 2018"        
[26] "#3093 - Messer 2002"        "#3185 - Montoya 2003"       "#3246 - Mukherjee 2022"     "#3293 - Myint 2006"         "#3469 - Nwe 2022"          
[31] "#3478 - Ocazionez 2006"     "#3552 - Osmali 2007"        "#365 - Aubry 2019"          "#3677 -

In [11]:
#Number of CovidenceIDs included in the main analysis (note that the study by Dussart is 
#split into two CovidenceIDs, despite being a single study). 
data_set %>% filter(N >= min_cases) %>% pull(CovidenceID) %>% unique %>% length

[1] 72

In [12]:
#Count of the number of study in each severity class system, for use in PRISMA flow diagram
data_set %>% select(SeverityType, CovidenceID) %>% distinct %>% group_by(SeverityType) %>% summarise(Count = n_distinct(CovidenceID))

# A tibble: 3 × 2
  SeverityType    Count
  <chr>           <int>
1 1997-type          42
2 2009-type          25
3 Hospitalisation    20

In [13]:
data_set %>% select(Name, SeverityType, CovidenceID) %>% distinct %>% filter(SeverityType == "2009-type") %>% arrange(Name)

# A tibble: 25 × 3
   Name                      SeverityType CovidenceID           
   <chr>                     <chr>        <chr>                 
 1 0002-Nwe_2022.xlsx        2009-type    #3469 - Nwe 2022      
 2 0017-Azevedo_2019.xlsx    2009-type    #395 - Azevedo 2019   
 3 0025-Sharp_2013_b.xlsx    2009-type    #4318 - Sharp 2013    
 4 0036-Huang_2020.xlsx      2009-type    #2084 - Huang 2020    
 5 0066-Martins_2014_a.xlsx  2009-type    #3017 - Martins 2014  
 6 0069a-Dussart_2012_a.xlsx 2009-type    #1379 - Dussart 2012a 
 7 0069b-Dussart_2012_b.xlsx 2009-type    #1379 - Dussart 2012b 
 8 0077-Mukherjee_2022.xlsx  2009-type    #3246 - Mukherjee 2022
 9 0083-Racherla_2018.xlsx   2009-type    #3868 - Racherla 2018 
10 0105-Yung_2015_b.xlsx     2009-type    #5345 - Yung 2015     
# ℹ 15 more rows
# ℹ Use `print(n = ...)` to see more rows

# 2. Data Processing

In [14]:
#This function takes in the desired severity class and then formats the data for the Stan model
generate_stan_input = function(curr_severity_class, unknown_included = TRUE, add_exclude = c()){
    #Get only the part of the dataset corresponding to the inputted severity class system
    sev_class_set = data_set %>% filter(SeverityType == curr_severity_class) %>% 
                    filter(!(CovidenceID %in% add_exclude)) #Additional study-specific exclusion filters based on CovidenceID
    
    #Check if the dataset has studies that contribute both specified and unspecified types. If so, discard the unspecified ones. 
    #Note that we count "ND" as a specified type.
    
    #Get CovidenceIDs of all studies with unspecified and specified types
    unspec_ids = sev_class_set %>% filter(PriorExp == "Unspecified") %>% pull(CovidenceID) %>% unname %>% unique
    spec_ids = sev_class_set %>% filter(PriorExp != "Unspecified") %>% pull(CovidenceID) %>% unname %>% unique
    
    #Check if there are studies with both - this check is important to prevent potentially double counting
    #Note that we count "ND" as a specified type (these were extracted to "mix" with the primary and secondary infections, but "Unspecified" cases do not)
    intersect_ids = intersect(unspec_ids, spec_ids)
    if(length(intersect_ids) > 0){print("Found Study with Specified and Unspecified Types", intersect_ids)}
    
    #Remove rows involving data with unspecified prior exposure, if the study is already giving cases with
    #specified prior exposure
    sev_class_set = sev_class_set %>% filter(!((CovidenceID %in% spec_ids) & (PriorExp == "Unspecified")))
    
    #After filtering studies, we can treat both ND and Unspecified as Unknown prior exposure status
    sev_class_set = sev_class_set %>% mutate(PriorExp = ifelse(PriorExp %in% c("Unspecified", "ND"), "Unknown", PriorExp))

    #Filter based on the minimum number of cases for the analysis (typically 5). We then add columns for the Scenario and Serotype-Prior exposure
    sev_class_set = sev_class_set %>% select(ID, CovidenceID, Region, Serotype, PriorExp, NonSevere, Severe, N) %>% 
                    filter(N >= min_cases) %>% 
                    mutate(SeroPriorExp = paste0(PriorExp, "-", Serotype)) %>%
                    mutate(Scenario = paste0(Region, "-", SeroPriorExp)) #Label each with scenario
  
    #If unknown_included is FALSE, remove rows where PriorExp == Unknown
    #With this, the analysis only includes primary or secondary cases
    if(!unknown_included){
        sev_class_set = sev_class_set %>% filter(PriorExp != "Unknown")
    }

    
    #We create a Scenario list to count the number of studies (plus the other main values) for each scenario
    sev_class_scenarios = sev_class_set %>% select(Scenario, CovidenceID, Severe, NonSevere, N) %>% group_by(Scenario) %>% 
                summarise(NumStudies = n_distinct(CovidenceID), Severe = sum(Severe), NonSevere = sum(NonSevere), N = sum(N))
    
    scenario_df = scenarios %>% left_join(sev_class_scenarios, by = "Scenario") %>% #Join to the full list of scenarios
                    mutate(across(everything(), ~ replace_na(., 0))) %>% #Replace the NAs with 0s
                    mutate(Inclusion = ifelse(NumStudies >= 2, "Included", "Excluded")) # We only include scenarios if they have at least two included studies
    
    #Add an index to each scenario. Excluded scenarios will have an index -1
    scenario_df = scenario_df %>% left_join(scenario_df %>% select(Scenario, Inclusion) %>% 
            filter(Inclusion == "Included") %>% mutate(ScenIndex = 1:nrow(.)), by = c("Scenario", "Inclusion")) %>%
            replace_na(list(ScenIndex = -1))
    
    #Check if there are studies that no longer have any outcomes left after discarding excluded scenarios
    pre_exc_ids = sev_class_set %>% pull(ID) %>% unique #IDs prior to exclusion
    excluded_scenarios = scenario_df %>% filter(Inclusion == "Excluded") %>% pull(Scenario) #Get excluded scenarios
    sev_class_set = sev_class_set %>% filter(!(Scenario %in% excluded_scenarios)) #Remove rows that are from excluded scenarios
    post_exc_ids = sev_class_set %>% pull(ID) %>% unique #Get the IDs post exclusion
    exclusion_check = pre_exc_ids[!(pre_exc_ids %in% post_exc_ids)] #Check if there were entire studies now excluded
    if(length(exclusion_check) > 0){print("Removed", exclusion_check)} #Print out any that were excluded
    
    
    #We start generating the necessary parameters
    n_row = sev_class_set %>% pull(ID) %>% unique %>% length #Number of studies in the analysis
    n_col = serotype_exposure %>% nrow #Usually 12 (4 serotypes, 3 prior exposure statuses)
    n_reg = length(regions) #Number of regions (usually 2, Asia and Americas)
    
    #Number of outcomes in the analysis (each outcome being a pair (n,e), corresponding to
    #the number of cases and severe cases for each study-region-serotype-prior exposure combination)
    n_outcomes = sev_class_set %>% nrow
    
    #Main data matrices, total and severe
    total = array(0, dim = c(n_row, n_col))
    severe = array(0, dim = c(n_row, n_col))
    
    scenario_indices = array(0, dim = c(n_reg, n_col)) #index of each scenario in each geographic region
    scenario_indices[1, ] = scenario_df %>% filter(Region == "Americas") %>% pull(ScenIndex) #Indices for Scenarios in Americas
    scenario_indices[2, ] = scenario_df %>% filter(Region == "Asia") %>% pull(ScenIndex)
    
    #Index vectors telling us the row, column, and region of each outcome part of the review
    row_indices = array(-1, dim = n_outcomes)
    col_indices = array(-1, dim = n_outcomes)
    reg_indices = array(-1, dim = n_outcomes)
    
    #DataFrame that counts the number of outcomes contributed by each study 
    outcome_counter = sev_class_set %>% select(ID, SeroPriorExp) %>% group_by(ID) %>% summarise(OutcomeCount = n_distinct(SeroPriorExp)) %>% ungroup
    
    #Make a DataFrame with 1 row per study, that then gives us the RowIndex and Region of each
    curr_study_df = sev_class_set %>% select(ID, CovidenceID, Region) %>% distinct %>% mutate(RowIndex = 1:nrow(.)) %>% 
                        left_join(region_df, by = "Region") %>% left_join(outcome_counter, by = "ID")
    
    #DataFrame of outcomes with relevant row, column, and region indices 
    curr_outcome_df = sev_class_set %>% left_join(curr_study_df %>% select(ID, RowIndex), by = c("ID")) %>% #join the row index
                            left_join(serotype_exposure %>% select(SeroPriorExp, ColIndex), by = "SeroPriorExp") %>% #join the column index
                            left_join(region_df, by = "Region") %>% arrange(RowIndex, ColIndex)#join the region index
    
    region_vals = curr_study_df %>% pull(RegIndex) #Region associated with each study
    num_results = curr_study_df %>% pull(OutcomeCount) #Number of outcomes for each study
    
    #Fill up both the main data matrices and the index vectors                
    for(i in 1:nrow(curr_outcome_df)){
        curr_row = curr_outcome_df[i, ]

        curr_row_ind = curr_row %>% pull(RowIndex)
        curr_col_ind= curr_row %>% pull(ColIndex)
        curr_reg_ind = curr_row %>% pull(RegIndex)
    
        curr_total = curr_row %>% pull(N)
        curr_severe = curr_row %>% pull(Severe)
    
        #Set the values in the total and severe matrix
        total[curr_row_ind, curr_col_ind] = curr_total
        severe[curr_row_ind, curr_col_ind] = curr_severe
        
        #Set index values
        row_indices[i] = curr_row_ind
        col_indices[i] = curr_col_ind
        reg_indices[i] = curr_reg_ind
    }
    
    #Number of scenarios (Region - Serotype - Prior Exposure) included in the analysis
    num_scenarios = scenario_df %>% filter(Inclusion == "Included") %>% nrow
    
    n_sero_prior = scenario_df %>% filter(Inclusion == "Included") %>% pull(SeroPriorExp) %>% unique %>% length
    
    #Create characteristic matrix
    #The characteristic matrix has number of rows equal to the number of outcomes
    #the number of columns is equal to the number of serotype-prior exposure comibnations included + 
    #the number of regions - 2 (since we take out the reference region and the reference serotype-prior exposure)
    #+1, because we let the first column be all ones as the intercept is always there
    
    char_matrix = array(0, dim = c(n_outcomes, 1 + (n_sero_prior-1) + (n_reg-1)))
    char_matrix[,1] = 1
    #Create a DataFrame so we know which characteristic matrix index each serotype-prior exposure combination corresponds to
    char_mat_guide = data.frame(SeroPriorExp = scenario_df %>% filter(Inclusion == "Included") %>% pull(SeroPriorExp) %>% unique) %>% 
                        arrange(SeroPriorExp)
    
    ref_ind = which(char_mat_guide$SeroPriorExp == ref_serotype_exposure) #Find which position the reference class belongs to
    
    char_mat_ind = 1:(n_sero_prior-1) %>% append(-1, after = (ref_ind -1)) #Create a seequence of numbers from 1 to n_sero_prior-1 then insert the -1 at the position of the reference class
    char_mat_guide = char_mat_guide %>% mutate(CharMatIndex = char_mat_ind) #Add the index of the column in the characteristic matrix that corresponds to each SeroPriorExp
    char_mat_reg_ind = char_matrix %>% ncol #Index of the column referring to the effect for geographic region (we assume here only 1 non-reference region, meaning it is the last index).
    
    #Fill the characteristic matrix based on each row of the outcome DF
    for(i in 1:nrow(curr_outcome_df)){
        curr_row = curr_outcome_df[i,]
        curr_SeroPriorExp = curr_row %>% pull(SeroPriorExp) #Get the column of the serotype prior exposure
        curr_Region = curr_row %>% pull(Region) #Get the region 
    
        char_mat_SeroPriorExp_ind = char_mat_guide %>% filter(SeroPriorExp == curr_SeroPriorExp) %>% pull(CharMatIndex) #get index corresponding to serotype prior exposure
        if(char_mat_SeroPriorExp_ind != -1){ #If -1 then it is the reference class and we leave SeroPriorExp columns as 0s
            #Have to offset by 1 due to the intercept column at the start
            char_matrix[i, char_mat_SeroPriorExp_ind + 1] = 1
        }
    
        if(curr_Region != ref_region){ #Set index based on the reference region
            char_matrix[i, char_mat_reg_ind] = 1
        }
    }
    
    reg_1_scen = scenario_df %>% filter(Region == "Americas", Inclusion == "Included") %>% nrow
    reg_2_scen = scenario_df %>% filter(Region == "Asia", Inclusion == "Included") %>% nrow
  
  
  
    #We create a mapper that converts 
    #ref_region = input_data$ref_region
    non_ref_region = ifelse(ref_region == "Asia", "Americas", "Asia")
    #ref_serotype_exposure = input_data$ref_serotype_exposure #Get reference serotype-prior exposure combination
    #ref_scenario = input_data$ref_scenario #Get reference scenario
    non_ref_seroprior = char_mat_guide %>% filter(CharMatIndex >0) %>% pull(SeroPriorExp) %>% as.character #Get all non-reference serotype-prior exposures

    #Get scenario labels ordered according to the model output which gives the reference scenario, then all non-reference serotype prior exposures in the reference region,
    #followed by the reference serotype-exposure in the non-reference region, and the rest of the serotype-prior exposures in the non-reference region
    scenario_labels = c(ref_scenario, #Reference scenario first
                        paste0(ref_region, "-", non_ref_seroprior), #Non-reference seropriors in the reference region
                        paste0(non_ref_region, "-", c(ref_serotype_exposure, non_ref_seroprior)) # Non-reference region with reference + non-reference seropriors
                    )

    #Add these indices to the scenario DataFrame from the model input
    reg_seroprior_inds = data.frame(Scenario = scenario_labels) %>% mutate(RegSeroPriorInd = 1:nrow(.))
    scenario_df = scenario_df %>% left_join(reg_seroprior_inds, by = "Scenario")
  

    scen_seroprior_ind_map = scenario_df %>% filter(Inclusion == "Included") %>% arrange(ScenIndex) %>% pull(RegSeroPriorInd)

    data_list = list(
        n_row = n_row, 
        n_col = n_col,
        n_reg = n_reg,
        n_outcomes = n_outcomes, 
        total = total,
        severe = severe,
        row_indices = row_indices,
        col_indices = col_indices,
        reg_indices = reg_indices,
        region_vals = region_vals,
        num_results = num_results,
        num_scenarios = num_scenarios, 
        n_sero_prior = n_sero_prior, 
        reg_1_scen = reg_1_scen,
        reg_2_scen = reg_2_scen,
        scenario_indices = scenario_indices,
        char_matrix = char_matrix,
        scen_seroprior_ind_map = scen_seroprior_ind_map
    )
    scenario_df = scenario_df %>% left_join(char_mat_guide, by = "SeroPriorExp") #Add the char matrix indices to the scenario_df
    
    return_list = list(data_list = data_list,
                      outcome_df = curr_outcome_df,
                      study_df = curr_study_df,
                      scenario_df = scenario_df,
                      char_mat_guide = char_mat_guide,
                      ref_region = ref_region,
                      ref_serotype_exposure = ref_serotype_exposure,
                      ref_scenario = ref_scenario)
}

In [15]:
#Generate the Stan input for each of the three severity classification systems
input_1997 = generate_stan_input("1997-type")
input_2009 = generate_stan_input("2009-type")
input_hosp = generate_stan_input("Hospitalisation")

In [16]:
input_1997_no_unknown = generate_stan_input("1997-type", unknown_included = FALSE)
input_2009_no_unknown = generate_stan_input("2009-type", unknown_included = FALSE)
input_hosp_no_unknown = generate_stan_input("Hospitalisation", unknown_included = FALSE)

In [17]:
#Here, we look at which studies are the most influential in the analysis of 1997-type severity
data_set %>% filter(!(PriorExp %in% c("Unspecified", "ND"))) %>% arrange(desc(N)) %>% filter(SeverityType == "1997-type")

#A manual check lists the following 
# #66694 - Narvaez 2025
# #4127 - Sabchareon 2012
# #1583 - Fried 2010

#We create some data sets excluding the influential studies here to use in our sensitivity analyses (in these analyses, unknowns are still included)
input_1997_no_narvaez = generate_stan_input("1997-type", add_exclude = c("#66694 - Narvaez 2025"))
input_1997_no_sabchareon = generate_stan_input("1997-type", add_exclude = c("#4127 - Sabchareon 2012"))
input_1997_no_fried = generate_stan_input("1997-type", add_exclude = c("#1583 - Fried 2010"))

In [18]:
data_set %>% filter(!(PriorExp %in% c("Unspecified", "ND"))) %>% arrange(desc(N)) %>% filter(SeverityType == "1997-type")

# A tibble: 100 × 11
   Serotype NonSevere Severe     N ID    CovidenceID             PriorExp  SeveritySystem SeverityType Region   Name                       
 * <chr>        <dbl>  <dbl> <dbl> <chr> <chr>                   <chr>     <chr>          <chr>        <chr>    <chr>                      
 1 DENV2          610    207   817 0331  #66694 - Narvaez 2025   Secondary 1997           1997-type    Americas 0331-Narvaez_2025a.xlsx    
 2 DENV3          331     56   387 0331  #66694 - Narvaez 2025   Primary   1997           1997-type    Americas 0331-Narvaez_2025a.xlsx    
 3 DENV3          282     88   370 0331  #66694 - Narvaez 2025   Secondary 1997           1997-type    Americas 0331-Narvaez_2025a.xlsx    
 4 DENV1          262     15   277 0331  #66694 - Narvaez 2025   Primary   1997           1997-type    Americas 0331-Narvaez_2025a.xlsx    
 5 DENV1          239     27   266 0331  #66694 - Narvaez 2025   Secondary 1997           1997-type    Americas 0331-Narvaez_2025a.xlsx    

In [ ]:
input_1997_no_unknown$outcome_df %>% 

# A tibble: 46 × 13
   ID    CovidenceID            Region   Serotype PriorExp  NonSevere Severe     N SeroPriorExp    Scenario                 RowIndex ColIndex RegIndex
   <chr> <chr>                  <chr>    <chr>    <chr>         <dbl>  <dbl> <dbl> <chr>           <chr>                       <int>    <int>    <int>
 1 0257  #3478 - Ocazionez 2006 Americas DENV2    Primary          10      1    11 Primary-DENV2   Americas-Primary-DENV2          1        6        1
 2 0257  #3478 - Ocazionez 2006 Americas DENV3    Primary          56      6    62 Primary-DENV3   Americas-Primary-DENV3          1        7        1
 3 0257  #3478 - Ocazionez 2006 Americas DENV2    Secondary         7     11    18 Secondary-DENV2 Americas-Secondary-DENV2        1       10        1
 4 0257  #3478 - Ocazionez 2006 Americas DENV3    Secondary        10      2    12 Secondary-DENV3 Americas-Secondary-DENV3        1       11        1
 5 0250  #1469 - Falconar 2012  Americas DENV2    Secondary         9     

In [19]:
#Output to RDS files, which we can read in later when we fit the model
if(save_output){
    saveRDS(input_1997, file.path(output_dir, "data_1997type.rds"))
    saveRDS(input_2009, file.path(output_dir, "data_2009type.rds"))
    saveRDS(input_hosp, file.path(output_dir, "data_hospitalisation.rds"))
  
    saveRDS(input_1997_no_unknown, file.path(output_dir, "data_1997type_no_unknown.rds"))
    saveRDS(input_2009_no_unknown, file.path(output_dir, "data_2009type_no_unknown.rds"))
    saveRDS(input_hosp_no_unknown, file.path(output_dir, "data_hospitalisation_no_unknown.rds"))
  
    saveRDS(input_1997_no_narvaez, file.path(output_dir, "data_1997type_no_narvaez.rds"))
    saveRDS(input_1997_no_sabchareon, file.path(output_dir, "data_1997type_no_sabchareon.rds"))
    saveRDS(input_1997_no_fried, file.path(output_dir, "data_1997type_no_fried.rds"))
}